In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import re

In [2]:
def plot_schedule(schedule, price, title, generator_names, color_map, starttime, n_generators=14, time_steps=168):
    hours = pd.date_range(start=starttime, periods=time_steps, freq="h")
    schedule = schedule.astype(int).to_numpy().reshape(n_generators, time_steps)
    status_by_hour = schedule.T

    hovertemplate = "".join(
        f"{name}: %{{customdata[{i}]}}<br>"
        for i, name in enumerate(generator_names)
    ) + "<extra></extra>"

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    for g, name in enumerate(generator_names):
        fig.add_trace(
            go.Bar(
                x=hours,
                y=schedule[g],
                opacity=0.7,
                name=name,
                hoverinfo="skip",
                marker_color=color_map[name]
            ),
            secondary_y=True,
        )

    fig.add_trace(
        go.Scatter(
            x=hours,
            y=np.zeros(time_steps),
            mode="markers",
            marker=dict(size=30, opacity=0),
            showlegend=False,
            hovertemplate=hovertemplate,
            customdata=status_by_hour,
            hoverinfo="skip",
        ),
        secondary_y=True,
    )

    fig.add_trace(
        go.Scatter(
            x=hours,
            y=price,
            mode="lines",
            name="Market price",
            line=dict(color="red"),
            hoverinfo="skip",
        ),
        secondary_y=False,
    )

    fig.update_layout(
        title=title,
        barmode="stack",
        xaxis_title="Date",
        yaxis_title="Price (EUR/MWh)",
        legend_title_text="Series",
        hovermode="closest",
        height=450,
    )

    fig.update_xaxes(tickformat="%Y-%m-%d %H:%M", showgrid=True)
    fig.update_yaxes(showgrid=True, title_text="Price (EUR/MWh)", secondary_y=False)
    fig.update_yaxes(showgrid=False, showticklabels=False, range=[0, n_generators], secondary_y=True)

    fig.show()

In [3]:
schedules = pd.read_csv('data/kernel/Generator_schedules_2015_2023.csv', index_col=0, parse_dates=['starttime'])
price_df = pd.read_csv('data/kernel/Historical_day_ahead_price_2015_2025.csv', parse_dates=['date']) 
price_df['date'] = price_df['date'].dt.tz_convert(None)

starttime = pd.Timestamp('2020-01-01') #select start date of schedule
endtime = starttime + pd.Timedelta(hours=168)

#get generator statuses for period
generator_cols = [col for col in schedules.columns if col.startswith('result_committed')] 
schedule = schedules.loc[schedules['starttime'] == starttime, generator_cols]

#get prices for period
price = price_df.loc[(price_df['date'] >= starttime) & (price_df['date'] < endtime),'Day-ahead price (EUR/MWh)'].to_numpy()

#extract generator names from the columns
generator_names = [] 
seen = set()
pattern = rf"^{re.escape('result_committed')}_(.+?_G\d+)_t\d+$"

for col in generator_cols:
    match = re.match(pattern, col)
    if match:
        name = match.group(1)
        if name not in seen:
            seen.add(name)
            generator_names.append(name)

In [4]:
palette = pc.qualitative.Dark24
color_map = {name: palette[i % len(palette)] for i, name in enumerate(generator_names)}
plot_schedule(schedule=schedule, price=price, title=f"Generator schedule starting {starttime.date()}", generator_names=generator_names, color_map=color_map, starttime=starttime)